### 1. Instalación de dependencias básicas
En esta celda instalamos `pandas`, necesario para la manipulación de datos.

In [ ]:
!pip install pandas

### 2. Descarga y filtrado del dataset
Este script descarga el dataset de noticias de la BBC, filtra los textos por longitud (entre 100 y 500 palabras) y guarda una muestra de 200 noticias.

In [ ]:
import pandas as pd

def generar_muestra_textos():
    # Enlace directo al CSV del dataset de noticias de la BBC en GitHub
    url = "https://raw.githubusercontent.com/suraj-deshmukh/BBC-Dataset-News-Classification/master/dataset/dataset.csv"

    print("Descargando el dataset desde GitHub...")
    try:
        # Leer el archivo CSV con codificación latin-1
        df = pd.read_csv(url, encoding='latin-1')
    except Exception as e:
        print(f"Error al descargar el archivo: {e}")
        return

    print("Procesando y contando palabras...")

    # En este dataset, la columna que contiene el texto se llama 'news'
    # Creamos una nueva columna con el conteo de palabras de cada texto
    df['word_count'] = df['news'].astype(str).apply(lambda x: len(x.split()))

    # Filtramos los textos para que tengan mínimo 100 y máximo 500 palabras
    df_filtrado = df[(df['word_count'] >= 100) & (df['word_count'] <= 500)]

    print(f"Textos que cumplen la condición (100-500 palabras): {len(df_filtrado)}")

    # Verificamos si tenemos al menos 200 textos
    if len(df_filtrado) >= 200:
        # Tomamos una muestra aleatoria de exactamente 200 textos
        # (random_state asegura que siempre obtengas la misma muestra si lo vuelves a ejecutar)
        muestra_final = df_filtrado.sample(n=200, random_state=42)

        nombre_archivo = "200_noticias_ingles.csv"

        # Guardamos solo la columna del texto en un nuevo archivo CSV, ignorando el índice
        muestra_final[['news']].to_csv(nombre_archivo, index=False)

        print("-" * 40)
        print(f"¡Éxito! Se guardó el archivo '{nombre_archivo}' en tu carpeta actual.")
        print("Resumen de tu muestra:")
        print(f"- Cantidad de textos: {len(muestra_final)}")
        print(f"- Promedio de palabras por texto: {muestra_final['word_count'].mean():.1f}")
        print(f"- Texto más corto: {muestra_final['word_count'].min()} palabras")
        print(f"- Texto más largo: {muestra_final['word_count'].max()} palabras")
        print("-" * 40)
    else:
        print("No hay suficientes textos que cumplan los requisitos de longitud.")

# Ejecutar la función
if __name__ == "__main__":
    generar_muestra_textos()

Descargando el dataset desde GitHub...
Procesando y contando palabras...
Textos que cumplen la condición (100-500 palabras): 1761
----------------------------------------
¡Éxito! Se guardó el archivo '200_noticias_ingles.csv' en tu carpeta actual.
Resumen de tu muestra:
- Cantidad de textos: 200
- Promedio de palabras por texto: 294.1
- Texto más corto: 122 palabras
- Texto más largo: 497 palabras
----------------------------------------


### 3. Instalación de Sentence Transformers
Instalamos la librería para generar embeddings vectoriales a partir de texto.

In [ ]:
!pip install -U sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 9.9 MB/s eta 0:00:00
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.4.1
    Uninstalling sentence-transformers-5.4.1:
      Successfully uninstalled sentence-transformers-5.4.1


### 4. Generación de Embeddings para las Noticias
Este script carga el modelo pre-entrenado, procesa los 200 textos y guarda los vectores resultantes en un archivo JSON.

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import json

def generar_embeddings():
    archivo_entrada = "200_noticias_ingles.csv"

    print("Cargando los textos...")
    try:
        df = pd.read_csv(archivo_entrada)
    except FileNotFoundError:
        print(f"Error: No se encontró '{archivo_entrada}'. Asegúrate de haber ejecutado el script anterior en esta misma sesión de Colab.")
        return

    # 1. Cargar el modelo Transformer
    print("Descargando/Cargando el modelo (puede tardar unos segundos la primera vez)...")
    modelo = SentenceTransformer('all-MiniLM-L6-v2')

    # 2. Convertir los textos a una lista
    textos = df['news'].tolist()

    # 3. Generar los embeddings
    print("Generando embeddings para los 200 textos...")
    # encode() procesa todo automáticamente y devuelve una matriz de numpy
    embeddings = modelo.encode(textos)

    # 4. Preparar la estructura para la Base de Datos
    # Añadimos un ID autoincremental, el texto y el vector asociado
    df_db = pd.DataFrame({
        'id': range(1, len(df) + 1),
        'texto': df['news'],
        'embedding': embeddings.tolist() # Se convierte a lista para que sea compatible con JSON/Bases de datos
    })

    # 5. Guardar los resultados
    # Exportamos a JSON. Es mucho más seguro y estándar que el CSV para importar arreglos de números (vectores) a una BD.
    archivo_salida = "../embeddings/news_embeddings.json"

    # orient='records' genera un formato tipo lista de diccionarios, ideal para MongoDB o APIs de BDs Vectoriales
    df_db.to_json(archivo_salida, orient='records', force_ascii=False, indent=4)

    print("-" * 40)
    print(f"¡Proceso completado exitosamente!")
    print(f"Se guardó el archivo: '{archivo_salida}'")
    print(f"Muestra procesada: {len(df_db)} registros")
    print(f"Dimensión de cada vector (embedding): {len(df_db['embedding'].iloc[0])} dimensiones")
    print("-" * 40)

# Ejecutar el script
generar_embeddings()

Cargando los textos...
Descargando/Cargando el modelo (puede tardar unos segundos la primera vez)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generando embeddings para los 200 textos...
----------------------------------------
¡Proceso completado exitosamente!
Se guardó el archivo: '../embeddings/news_embeddings.json'
Muestra procesada: 200 registros
Dimensión de cada vector (embedding): 384 dimensiones
----------------------------------------


### 5. Generación de Embeddings para Consultas (Queries)
Aquí definimos las 8 consultas de prueba y generamos sus vectores correspondientes para usarlos en búsquedas semánticas.

In [8]:
from sentence_transformers import SentenceTransformer
import json

def generar_embeddings_consultas():
    # 1. Definimos las 8 consultas propuestas
    consultas = [
        "The impact of rising inflation and interest rates on the economy",
        "A deep feeling of sadness, tragedy, and emotional loss",
        "Wild animals, endangered species, and nature conservation",
        "Things are getting worse very quickly and people are complaining",
        "Who won the best actor award at the film festival?",
        "Next generation digital software and mobile internet innovations",
        "Election parliament prime minister voting campaign",
        "Tropical fruits like apples, bananas, and oranges"
    ]

    # 2. Cargamos el MISMO modelo que usamos para los textos (obligatorio)
    print("Cargando el modelo 'all-MiniLM-L6-v2'...")
    modelo = SentenceTransformer('all-MiniLM-L6-v2')

    # 3. Generamos los embeddings
    print("Generando vectores para las 8 consultas...")
    embeddings_consultas = modelo.encode(consultas)

    # 4. Estructuramos los datos para guardarlos
    datos_exportar = []
    for i, (texto, vector) in enumerate(zip(consultas, embeddings_consultas)):
        datos_exportar.append({
            "id_consulta": i + 1,
            "texto_consulta": texto,
            "embedding": vector.tolist()  # Convertimos a lista para que sea compatible con JSON
        })

    # 5. Guardamos en formato JSON
    archivo_salida = "../embeddings/query_embeddings.json"
    with open(archivo_salida, 'w', encoding='utf-8') as f:
        json.dump(datos_exportar, f, ensure_ascii=False, indent=4)

    print("-" * 50)
    print(f"¡Éxito! Se generó el archivo: '{archivo_salida}'")
    print(f"Contiene las {len(datos_exportar)} consultas con sus vectores de 384 dimensiones.")
    print("-" * 50)

# Ejecutar la función
generar_embeddings_consultas()

Cargando el modelo 'all-MiniLM-L6-v2'...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generando vectores para las 8 consultas...
--------------------------------------------------
¡Éxito! Se generó el archivo: '../embeddings/query_embeddings.json'
Contiene las 8 consultas con sus vectores de 384 dimensiones.
--------------------------------------------------


### 6. Creación del Script SQL
Este script genera el archivo `.sql` completo con la creación de tablas y todos los comandos `INSERT` necesarios para cargar noticias y consultas en Oracle Database.

In [13]:
import pandas as pd
from sentence_transformers import SentenceTransformer

def generar_script_final():
    archivo_entrada = "200_noticias_ingles.csv"
    archivo_salida_sql = "../sql/generated_inserts_schema_vectorial.sql"

    print("1. Cargando los textos desde el CSV...")
    try:
        df = pd.read_csv(archivo_entrada)
    except FileNotFoundError:
        print(f"Error: No se encontró '{archivo_entrada}'.")
        return

    print("2. Cargando el modelo 'all-MiniLM-L6-v2'...")
    modelo = SentenceTransformer('all-MiniLM-L6-v2')

    print("3. Generando embeddings de los 200 textos...")
    textos = df['news'].tolist()
    embeddings_textos = modelo.encode(textos)

    print("4. Generando embeddings de las 8 consultas...")
    consultas = [
        ("Query_Economia", "The impact of rising inflation and interest rates on the economy"),
        ("Query_Emocion", "A deep feeling of sadness, tragedy, and emotional loss"),
        ("Query_Animales", "Wild animals, endangered species, and nature conservation"),
        ("Query_Vaga", "Things are getting worse very quickly and people are complaining"),
        ("Query_Evento", "Who won the best actor award at the film festival?"),
        ("Query_Tecnologia", "Next generation digital software and mobile internet innovations"),
        ("Query_Politica", "Election parliament prime minister voting campaign"),
        ("Query_Frutas", "Tropical fruits like apples, bananas, and oranges")
    ]
    textos_consultas = [q[1] for q in consultas]
    embeddings_consultas = modelo.encode(textos_consultas)

    print("5. Armando el archivo SQL...")
    with open(archivo_salida_sql, 'w', encoding='utf-8') as f:
        # Configuración inicial para evitar el problema del símbolo &
        f.write("SET DEFINE OFF;\n\n")

        # Creación de Tablas (Sin las columnas title ni topic)
        f.write("-- Oracle 23ai Free  –  Vector Search schema\n")
        f.write("DROP TABLE IF EXISTS news_articles;\n")
        f.write("DROP TABLE IF EXISTS query_vectors;\n\n")

        f.write("CREATE TABLE news_articles (\n")
        f.write("    id        NUMBER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,\n")
        f.write("    content   CLOB,\n")
        f.write("    embedding VECTOR(384, FLOAT32)\n")
        f.write(");\n\n")

        f.write("CREATE TABLE query_vectors (\n")
        f.write("    id         NUMBER PRIMARY KEY,\n")
        f.write("    label      VARCHAR2(200),\n")
        f.write("    query_text VARCHAR2(500),\n")
        f.write("    embedding  VECTOR(384, FLOAT32)\n")
        f.write(");\n\n")

        # Inserción de Noticias (Solo content y embedding)
        f.write("-- Inserción de las 200 Noticias\n")
        for i, (texto, vector) in enumerate(zip(textos, embeddings_textos)):
            # Escapar comillas simples
            texto_limpio = texto.replace("'", "''")

            # Redondeamos el vector a 5 decimales para evitar el error ORA-01704
            vector_redondeado = [round(float(v), 5) for v in vector]
            vector_str = "[" + ",".join(map(str, vector_redondeado)) + "]"

            # El campo ID se omite por ser IDENTITY
            sql = f"INSERT INTO news_articles (content, embedding) VALUES ('{texto_limpio}', '{vector_str}');\n"
            f.write(sql)

        f.write("\n-- Inserción de las 8 Consultas\n")
        for i, (datos_query, vector) in enumerate(zip(consultas, embeddings_consultas)):
            id_query = i + 1
            label = datos_query[0]
            texto_query_limpio = datos_query[1].replace("'", "''")

            vector_redondeado = [round(float(v), 5) for v in vector]
            vector_str = "[" + ",".join(map(str, vector_redondeado)) + "]"

            sql = f"INSERT INTO query_vectors (id, label, query_text, embedding) VALUES ({id_query}, '{label}', '{texto_query_limpio}', '{vector_str}');\n"
            f.write(sql)

        f.write("\nCOMMIT;\n")

    print("-" * 50)
    print(f"¡Éxito! Se generó el script súper limpio: '{archivo_salida_sql}'")
    print("Ya puedes descargarlo y ejecutarlo en Oracle Database Actions.")
    print("-" * 50)

# Ejecutar la función
generar_script_final()

1. Cargando los textos desde el CSV...
2. Cargando el modelo 'all-MiniLM-L6-v2'...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


3. Generando embeddings de los 200 textos...
4. Generando embeddings de las 8 consultas...
5. Armando el archivo SQL...
--------------------------------------------------
¡Éxito! Se generó el script súper limpio: '../sql/generated_inserts_schema_vectorial.sql'
Ya puedes descargarlo y ejecutarlo en Oracle Database Actions.
--------------------------------------------------
